# Testes Iniciais — Avaliação CEP + ML
## Classificação de Variedades de Grãos de Feijão (Dry Bean Dataset)

**Autora:** Maria Eduarda Lobo Montenegro
**Disciplina:** Controle Estatístico de Processos — UnB
**Professor:** Andre Luiz Marques Serrano
**Data:** Abril de 2026
**Entrega:** Parte 1 — Testes Iniciais (carga + EDA básica)

---

### Objetivo desta entrega parcial
Demonstrar que o pipeline inicial do projeto está funcional no Google Colab:
- Carregamento do dataset escolhido (Dry Bean — UCI Repository)
- Inspeção da estrutura dos dados
- Distribuição da variável-alvo (7 variedades)
- Estatísticas descritivas das 16 features morfológicas
- Visualizações exploratórias iniciais

### Dataset
**Dry Bean Dataset** — UCI Machine Learning Repository (id=602)
- 13.611 grãos × 16 features morfológicas + 1 target (7 variedades)
- Fonte: https://archive.ics.uci.edu/dataset/602/dry+bean+dataset

### Próximas entregas
- Parte 2: CEP completo (cartas X-barra/R + Cp/Cpk)
- Parte 3: Modelagem (LogReg, Random Forest, SVM) + Otimização
- Parte 4: Relatório técnico final


## 0. Instalação de dependências

In [ ]:
# Instalação do pacote oficial do UCI Repository
!pip install -q ucimlrepo

In [ ]:
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

# Configuração visual
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
np.random.seed(42)

print('Bibliotecas carregadas com sucesso.')

---
## 1. Carregamento do Dataset

Carregamos o **Dry Bean Dataset** diretamente do UCI Machine Learning Repository via pacote oficial `ucimlrepo`. Isso garante 100% de reprodutibilidade — qualquer execução obtém os mesmos dados.

In [ ]:
# Download direto do repositório UCI (id=602)
dry_bean = fetch_ucirepo(id=602)

# Construção do DataFrame consolidado (features + target)
X = dry_bean.data.features
y = dry_bean.data.targets
df = pd.concat([X, y], axis=1)

print(f'Dataset carregado: {df.shape[0]} amostras x {df.shape[1]} colunas')
print(f'Features: {X.shape[1]}')
print(f'Classes (target): {y.iloc[:, 0].nunique()}')

---
## 2. Inspeção Inicial da Estrutura

Conferimos tipos de dados, valores ausentes e amostra das primeiras linhas.

In [ ]:
# Informações estruturais
print('=== INFO ===')
df.info()
print()
print('=== VALORES AUSENTES POR COLUNA ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Nenhum valor ausente — dataset completo.')

In [ ]:
# Primeiras 5 amostras
df.head()

**Observações iniciais:**
- Dataset **completo** (sem valores ausentes)
- 16 features numéricas (`float64`/`int64`) + 1 coluna categórica (`Class`)
- Pronto para análise — não há limpeza estrutural necessária

---
## 3. Distribuição da Variável-Alvo (Target)

A target representa 7 variedades comerciais de feijão. Verificamos o **balanceamento** das classes — fator crítico para a escolha de métricas de avaliação do modelo de ML.

In [ ]:
# Distribuição das classes
target_col = y.columns[0]
class_counts = df[target_col].value_counts()
class_pct = (class_counts / len(df) * 100).round(2)

dist = pd.DataFrame({'Contagem': class_counts, 'Proporção (%)': class_pct})
print(dist)
print(f'\nTotal: {len(df)} amostras')

In [ ]:
# Visualização da distribuição
fig, ax = plt.subplots(figsize=(11, 5))
colors = sns.color_palette('viridis', n_colors=len(class_counts))
bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor='black')

for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{count}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Distribuição das 7 Variedades de Feijão', fontsize=13, fontweight='bold')
ax.set_xlabel('Variedade')
ax.set_ylabel('Número de Amostras')
plt.tight_layout()
plt.show()

**Diagnóstico de balanceamento:**
- **Classe majoritária:** DERMASON (~26%)
- **Classe minoritária:** BOMBAY (~3,8%)
- **Conclusão:** classes **desbalanceadas** → na fase de modelagem, será usada **F1-macro** como métrica principal (trata cada classe com igual importância, independente do tamanho)

---
## 4. Estatísticas Descritivas das 16 Features

Resumo numérico das features morfológicas — média, desvio, mínimos, máximos. Calculamos também o **Coeficiente de Variação (CV%)** para identificar features com maior variabilidade relativa.

In [ ]:
# Estatísticas descritivas
stats = df.describe().T[['mean', 'std', 'min', 'max']]
stats['CV (%)'] = (stats['std'] / stats['mean'] * 100).round(2)
stats = stats.round(3)
stats.columns = ['Média', 'Desvio Padrão', 'Mínimo', 'Máximo', 'CV (%)']
stats.sort_values('CV (%)', ascending=False)

**Insights iniciais:**
- **Maior variabilidade:** `Area` e `ConvexArea` (CV ≈ 55%) — diferenças dimensionais grandes entre variedades (BOMBAY é muito maior)
- **Menor variabilidade:** `Solidity` e `ShapeFactor4` (CV < 1%) — propriedades estruturais muito uniformes
- Espera-se **assimetria à direita** em Area/Perimeter pela presença de BOMBAY

---
## 5. Análise Exploratória Visual

### 5.1 Distribuição das principais features dimensionais

In [ ]:
# Histogramas de 6 features-chave
features_plot = ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity']
# Trata grafia alternativa (UCI usa "AspectRation")
features_plot = [f for f in features_plot if f in df.columns]
if 'AspectRatio' in df.columns and 'AspectRation' not in df.columns:
    features_plot = ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRatio', 'Eccentricity']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), features_plot):
    ax.hist(df[feat], bins=40, color='steelblue', edgecolor='black', alpha=0.75)
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Frequência')
plt.tight_layout()
plt.show()

### 5.2 Matriz de correlação (Pearson)

In [ ]:
# Heatmap de correlações
plt.figure(figsize=(12, 9))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
plt.title('Matriz de Correlação — 16 Features Morfológicas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Multicolinearidade detectada (|r| > 0,90):**
- `Area` ↔ `ConvexArea` (r ≈ 1,00) — redundância dimensional
- `Area` ↔ `EquivDiameter` (r ≈ 0,99)
- `Perimeter` ↔ `MajorAxisLength` (r ≈ 0,97)
- `Eccentricity` ↔ `AspectRatio` (r ≈ 0,96)

**Implicação:** será necessário cuidado com modelos sensíveis à multicolinearidade (Regressão Logística). Modelos baseados em árvores (Random Forest) são robustos a esse problema.

---
## 6. Resumo dos Testes Iniciais

| Verificação | Status |
|---|:---:|
| Carregamento do dataset via `ucimlrepo` | ✅ OK |
| 13.611 amostras × 17 colunas | ✅ OK |
| Sem valores ausentes | ✅ OK |
| 7 classes identificadas | ✅ OK |
| Classes desbalanceadas (DERMASON 26% vs BOMBAY 3,8%) | ✅ Mapeado |
| Estatísticas descritivas calculadas | ✅ OK |
| Visualizações geradas (distribuição, histogramas, correlação) | ✅ OK |
| Multicolinearidade identificada | ✅ Mapeado |

### Conclusão
O **pipeline inicial está funcional** e pronto para avançar para as próximas etapas do projeto:

1. **Próxima entrega (Parte 2):** Controle Estatístico de Processos
   - Cartas de Controle X-barra e R (n=5)
   - Índices de Capacidade Cp, Cpk, Pp, Ppk
   - Diagnóstico de estabilidade e capacidade

2. **Entregas posteriores:** preparação de dados, modelagem (LogReg, Random Forest, SVM), otimização via GridSearchCV, avaliação final e relatório técnico completo.

---

*Notebook executado em Google Colab — Python 3.10+*
*Repositório: https://github.com/melobomontenegro-dev/CEP_Projeto_Maria_Eduarda*